# Func_apxm_MLP: xấp xỉ hàm Gaussian 2D bằng MLP (chuyển từ Keras sang PyTorch)

MLP 2 → B (sigmoid) → 1 (sigmoid), loss MSE, SGD lr = 0.01, batch 128. Ba thí nghiệm: thay đổi số mẫu huấn luyện A, số nút ẩn B, số epoch C.

Mạng rất nhỏ nên chạy trên **CPU** cho nhanh (GPU không giúp được gì ở cỡ này).

# (1) Thư viện và thiết lập

In [ ]:
import os, random
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import multivariate_normal
from google.colab import drive

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/BTL03'
os.makedirs(BASE + '/P1', exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cpu')          # mạng nhỏ: CPU nhanh hơn GPU
print(torch.__version__, device)

# (2) Chuẩn bị dữ liệu

Hàm Gaussian 2D (mu = 0, covariance = đơn vị) trên lưới [-3, 3] × [-3, 3]. Train: lưới 32 × 32 = 1024 điểm. Test: lưới 10 × 10 = 100 điểm. Phần này không phụ thuộc Keras nên giữ nguyên như bản gốc.

In [ ]:
def make_data(n):
    x1, x2 = np.mgrid[-3.0:3.0:complex(0, n), -3.0:3.0:complex(0, n)]   # tương đương 32j, 10j
    x1x2 = np.column_stack([x1.flat, x2.flat])
    y = multivariate_normal.pdf(x1x2, mean=np.array([0.0, 0.0]), cov=np.diag([1, 1]))
    return x1, x2, x1x2, y.reshape(x1.shape)

x1, x2, x_train, y_surf = make_data(32); y_train = y_surf.flatten()
fig = plt.figure(); ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(x1, x2, y_surf); plt.title('training data'); plt.show()

x1t, x2t, x_test, y_surf_t = make_data(10); y_test = y_surf_t.flatten()
fig = plt.figure(); ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(x1t, x2t, y_surf_t); plt.title('testing data'); plt.show()

# chuyển sang tensor (nhãn có dạng (N, 1) để khớp đầu ra của mô hình)
to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
X_TR, Y_TR = to_t(x_train), to_t(y_train).unsqueeze(1)
X_TE, Y_TE = to_t(x_test), to_t(y_test).unsqueeze(1)
print(X_TR.shape, Y_TR.shape, X_TE.shape, Y_TE.shape)   # (1024,2) (1024,1) (100,2) (100,1)

# (3) MLP

Keras: `Input(2)` → `Dense(B, sigmoid)` → `Dense(1, sigmoid)`. PyTorch: hai lớp `nn.Linear`, mỗi lớp qua sigmoid. Keras khởi tạo trọng số Glorot-uniform và bias 0, nên mình đặt lại cho giống (PyTorch mặc định dùng cách khởi tạo khác).

In [ ]:
class GaussMLP(nn.Module):
    def __init__(self, B=16):
        super().__init__()
        self.fc = nn.Linear(2, B)
        self.out = nn.Linear(B, 1)

    def forward(self, x):
        x = torch.sigmoid(self.fc(x))
        return torch.sigmoid(self.out(x))

def init_keras_like(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

m = GaussMLP(16)
print('Số tham số (B=16):', sum(p.numel() for p in m.parameters()))   # phải là 65

# (4) Huấn luyện và đánh giá

`model.compile + model.fit` của Keras được thay bằng vòng lặp viết tay: mỗi epoch xáo trộn dữ liệu, chia batch 128, rồi `zero_grad → forward → loss → backward → step`. `model.evaluate` trả về MSE trung bình, tương đương `nn.MSELoss`.

In [ ]:
loss_fn = nn.MSELoss()

def train_model(x, y, B, epochs, lr=0.01, batch_size=128):
    model = GaussMLP(B).to(device)
    model.apply(init_keras_like)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    n = x.shape[0]
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            loss = loss_fn(model(x[idx]), y[idx])
            loss.backward()
            opt.step()
    return model

@torch.no_grad()
def mse(model, x, y):
    return loss_fn(model(x), y).item()

TOT_TRAIN = 1001   # giữ nguyên như bản gốc: chọn chỉ số trong range(1, 1001)

def run_sweep(values, get_cfg):
    tr_err, te_err = [], []
    for v in values:
        A, B, C = get_cfg(v)
        sel = np.random.choice(range(1, TOT_TRAIN), size=A, replace=False)
        sel = torch.as_tensor(sel, device=device)
        model = train_model(X_TR[sel], Y_TR[sel], B, C)
        tr_err.append(mse(model, X_TR, Y_TR))    # như bản gốc: đánh giá trên toàn bộ 1024 điểm train
        te_err.append(mse(model, X_TE, Y_TE))
        print(f"{v:>4} | train {tr_err[-1]:.6f} | test {te_err[-1]:.6f}")
    return tr_err, te_err

def plot_sweep(values, tr, te, title, xlabel, fname):
    plt.figure(figsize=(6, 4))
    plt.plot(values, tr, label='training'); plt.plot(values, te, label='testing')
    plt.legend(); plt.title(title); plt.xlabel(xlabel); plt.ylabel('Error')
    plt.tight_layout(); plt.savefig(BASE + '/P1/' + fname, dpi=150); plt.show()

### (4.1) Số mẫu huấn luyện A (số nút ẩn B = 16, số epoch C = 100)

In [ ]:
A_values = np.arange(100, TOT_TRAIN, 100)
tr41, te41 = run_sweep(A_values, lambda v: (v, 16, 100))
plot_sweep(A_values, tr41, te41, 'Number of training samples (A)', '#-samples', 'func_apxm_A.png')

### (4.2) Số nút ẩn B (A = 800, C = 100)

In [ ]:
B_values = [8, 16, 32, 64, 128, 256, 512]
tr42, te42 = run_sweep(B_values, lambda v: (800, v, 100))
plot_sweep(B_values, tr42, te42, 'Number of hidden nodes (B)', 'Number of hidden nodes', 'func_apxm_B.png')

### (4.3) Số epoch C (A = 800, B = 128)

In [ ]:
C_values = np.arange(10, 200, 10)
tr43, te43 = run_sweep(C_values, lambda v: (800, 128, v))
plot_sweep(C_values, tr43, te43, 'Number of epochs (C)', 'Epochs', 'func_apxm_C.png')

# Nhận xét (điền sau khi có kết quả)

- Thí nghiệm A: …
- Thí nghiệm B: …
- Thí nghiệm C: …

# Điểm khác biệt Keras → PyTorch trong notebook này

- `Model(Input → Dense → Dense)` → lớp `nn.Module` với `nn.Linear`; sigmoid gọi trong `forward`.
- `compile + fit` → vòng lặp huấn luyện viết tay (`randperm` để xáo trộn, chia batch 128).
- `evaluate` → `nn.MSELoss` trong `torch.no_grad()`.
- Khởi tạo: Keras Glorot-uniform + bias 0, PyTorch mặc định khác, nên đã đặt lại bằng `xavier_uniform_` và `zeros_`.
- Bản gốc chọn mẫu bằng `range(1, 1001)` nên chỉ số 0 không bao giờ được chọn; giữ nguyên để kết quả so sánh được.